In [2]:
import copy
import csv
import os
import warnings
from argparse import ArgumentParser
import numpy as np
import torch
from tqdm import tqdm
import yaml
from torch.utils import data
# 개별 json 라벨 파일을 이용해 학습 데이터 리스트 생성
import glob
import json
import os
import pandas as pd
from PIL import Image
from torch.utils import data
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:",device)

device: cuda:0


In [3]:
# M1 TCGA-only 데이터 로드
# 입력 단위: 1 slide(case)의 tile image paths + tile 좌표 + slide 전체 크기 + multi-horizon survival label/mask
# 출력 label: dead by 6/12/18/24 months. Unknown(censored before horizon)은 mask=0으로 loss에서 제외합니다.

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DATA_PATH = Path("../../data")
PROJECT_DATA_PATH = DATA_PATH / "pancreatic_cancer_pathology"
DST_PATH = PROJECT_DATA_PATH / "dst"
IMAGE_PATH = DST_PATH / "Image"
CLINICAL_PATH = DST_PATH / "Clinical"

M1_OUTPUT_PATH = Path("outputs/M1")
M1_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "TCGA_PAAD"
SEED = 42
PATCH_INPUT_SIZE = 512  # 1.0 MPP 1024 tile을 512 입력으로 줄이면 effective MPP는 약 2.0입니다.
TEST_SIZE = 0.2
VALID_SIZE = 0.2

MONTH_DAYS = 30.4375
HORIZON_MONTHS = [6, 12, 18, 24]
HORIZON_DAYS = np.array([m * MONTH_DAYS for m in HORIZON_MONTHS], dtype=np.float32)
HORIZON_NAMES = [f"dead_by_{m}m" for m in HORIZON_MONTHS]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def load_clinical_label(dataset: str, case_id: str) -> tuple[float | None, int | None]:
    clinical_json_path = CLINICAL_PATH / dataset / f"{case_id}_clinical.json"
    if not clinical_json_path.exists():
        return None, None
    with open(clinical_json_path, "r", encoding="utf-8") as f:
        clinical_json = json.load(f)
    clinical = clinical_json.get("clinical", {})
    return clinical.get("os_time_days"), clinical.get("os_event")


def make_horizon_label_mask(os_time_days: float, os_event: int) -> tuple[np.ndarray, np.ndarray]:
    """Create multi-horizon dead-by labels and known-label mask.

    label[h] = 1 if death occurred by horizon h, else 0.
    mask[h] = 1 if the label at horizon h is known.

    Censored before a horizon is unknown and excluded from BCE loss with mask=0.
    """
    y = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    mask = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    if pd.isna(os_time_days) or pd.isna(os_event):
        return y, mask

    os_time_days = float(os_time_days)
    os_event = int(os_event)

    for i, horizon in enumerate(HORIZON_DAYS):
        if os_event == 1 and os_time_days <= float(horizon):
            y[i] = 1.0
            mask[i] = 1.0
        elif os_time_days >= float(horizon):
            y[i] = 0.0
            mask[i] = 1.0
        else:
            # Censored before this horizon: unknown.
            y[i] = 0.0
            mask[i] = 0.0
    return y, mask


def get_train_patch_transform(image_size: int = PATCH_INPUT_SIZE):
    # Resize를 augmentation보다 먼저 수행해 1024 tile에서 바로 연산하지 않도록 합니다.
    # UNI/UNI v2 mean/std 및 512 입력 허용 여부는 feature extractor 로드 셀에서 확인합니다.
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)], p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.15),
        transforms.ToTensor(),
    ])


def get_eval_patch_transform(image_size: int = PATCH_INPUT_SIZE):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.ToTensor(),
    ])


def load_tcga_case_tiles(summary_row: pd.Series) -> tuple[pd.DataFrame, dict]:
    case_id = str(summary_row["case_id"])
    case_dir = Path(summary_row["case_dir"])
    metadata_path = Path(summary_row["metadata_path"])
    tile_df = pd.read_csv(metadata_path)

    tile_df["x"] = tile_df["x_level0"].astype(int)
    tile_df["y"] = tile_df["y_level0"].astype(int)
    slide_width = int(summary_row["slide_width"])
    slide_height = int(summary_row["slide_height"])

    tile_df["tile_path"] = tile_df["tile_path"].astype(str)
    tile_df = tile_df[tile_df["tile_path"].map(lambda x: Path(x).exists())].copy()
    tile_df["slide_width"] = slide_width
    tile_df["slide_height"] = slide_height

    source_size = tile_df["source_tile_size_px"].astype(float)
    tile_df["x_norm"] = tile_df["x"].astype(float) / slide_width
    tile_df["y_norm"] = tile_df["y"].astype(float) / slide_height
    tile_df["x_center_norm"] = (tile_df["x"].astype(float) + source_size / 2) / slide_width
    tile_df["y_center_norm"] = (tile_df["y"].astype(float) + source_size / 2) / slide_height
    tile_df["w_norm"] = source_size / slide_width
    tile_df["h_norm"] = source_size / slide_height

    os_time, os_event = load_clinical_label(DATASET_NAME, case_id)
    if os_time is None or os_event is None:
        os_time = tile_df["OS_time"].iloc[0] if "OS_time" in tile_df.columns else np.nan
        os_event = tile_df["OS_event"].iloc[0] if "OS_event" in tile_df.columns else np.nan

    y, mask = make_horizon_label_mask(os_time, os_event)
    case_record = {
        "dataset": DATASET_NAME,
        "case_id": case_id,
        "case_dir": case_dir.as_posix(),
        "metadata_path": metadata_path.as_posix(),
        "n_tiles": int(len(tile_df)),
        "slide_width": slide_width,
        "slide_height": slide_height,
        "os_time_days": float(os_time) if pd.notna(os_time) else np.nan,
        "os_event": int(os_event) if pd.notna(os_event) else np.nan,
        "known_horizon_count": int(mask.sum()),
    }
    for i, name in enumerate(HORIZON_NAMES):
        case_record[name] = float(y[i])
        case_record[f"mask_{name}"] = float(mask[i])
    return tile_df, case_record


summary_path = IMAGE_PATH / DATASET_NAME / "tile_summary.csv"
summary_df = pd.read_csv(summary_path)
summary_df = summary_df[summary_df["metadata_path"].notna()].copy()

case_records = []
tile_index_list = []

for _, row in tqdm(summary_df.iterrows(), total=len(summary_df), desc=f"Load {DATASET_NAME} metadata"):
    metadata_path = Path(row["metadata_path"])
    if not metadata_path.exists():
        continue
    tile_df, case_record = load_tcga_case_tiles(row)
    if case_record["n_tiles"] == 0:
        case_record["exclude_reason"] = "no_tiles"
        case_records.append(case_record)
        continue
    tile_df["dataset"] = DATASET_NAME
    tile_df["case_id"] = case_record["case_id"]
    tile_index_list.append(tile_df)
    case_records.append(case_record)

all_slide_df = pd.DataFrame(case_records)
tile_index_df = pd.concat(tile_index_list, ignore_index=True)

eligible_mask = (
    all_slide_df["os_time_days"].notna()
    & all_slide_df["os_event"].notna()
    & all_slide_df["known_horizon_count"].gt(0)
    & all_slide_df["n_tiles"].gt(0)
)
excluded_df = all_slide_df[~eligible_mask].copy()
if not excluded_df.empty:
    excluded_df["exclude_reason"] = np.select(
        [
            excluded_df["n_tiles"].le(0),
            excluded_df["os_time_days"].isna(),
            excluded_df["os_event"].isna(),
            excluded_df["known_horizon_count"].le(0),
        ],
        ["no_tiles", "missing_os_time", "missing_os_event", "no_known_horizon"],
        default="unknown",
    )

slide_df = all_slide_df[eligible_mask].copy()
slide_df["os_event"] = slide_df["os_event"].astype(int)
tile_index_df = tile_index_df[
    tile_index_df["case_id"].isin(slide_df["case_id"])
    & tile_index_df["dataset"].eq(DATASET_NAME)
].copy()

slide_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_horizon_slide_manifest.csv", index=False)
tile_index_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_horizon_tile_index.csv", index=False)
excluded_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_horizon_excluded_cases.csv", index=False)

print("all_slide_df:", all_slide_df.shape)
print("slide_df:", slide_df.shape)
print("excluded_df:", excluded_df.shape)
print("tile_index_df:", tile_index_df.shape)

horizon_summary = []
for name in HORIZON_NAMES:
    mask_col = f"mask_{name}"
    known = slide_df[mask_col].eq(1)
    horizon_summary.append({
        "horizon": name,
        "known_n": int(known.sum()),
        "dead_n": int(slide_df.loc[known, name].sum()),
        "alive_n": int(known.sum() - slide_df.loc[known, name].sum()),
        "unknown_n": int((~known).sum()),
        "dead_rate_known": float(slide_df.loc[known, name].mean()) if known.any() else np.nan,
    })
horizon_summary_df = pd.DataFrame(horizon_summary)
display(horizon_summary_df)
display(slide_df[["case_id", "n_tiles", "os_time_days", "os_event", "known_horizon_count"] + HORIZON_NAMES + [f"mask_{n}" for n in HORIZON_NAMES]].head())


class M1SlideDataset(Dataset):
    """TCGA case-level pathology dataset for multi-horizon MIL training."""

    def __init__(self, slide_manifest: pd.DataFrame, tile_index: pd.DataFrame):
        self.slide_manifest = slide_manifest.reset_index(drop=True).copy()
        self.tile_groups = {
            case_id: group.sort_values(["y", "x"]).reset_index(drop=True)
            for case_id, group in tile_index.groupby("case_id")
        }

    def __len__(self):
        return len(self.slide_manifest)

    def __getitem__(self, idx):
        row = self.slide_manifest.iloc[idx]
        tiles = self.tile_groups[row["case_id"]]

        coords = tiles[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        label = row[HORIZON_NAMES].to_numpy(np.float32)
        mask = row[[f"mask_{name}" for name in HORIZON_NAMES]].to_numpy(np.float32)
        slide_size = np.array([row["slide_width"], row["slide_height"]], dtype=np.float32)

        return {
            "dataset": row["dataset"],
            "case_id": row["case_id"],
            "tile_paths": tiles["tile_path"].tolist(),
            "coords": torch.from_numpy(coords),
            "slide_size": torch.from_numpy(slide_size),
            "horizon_months": torch.tensor(HORIZON_MONTHS, dtype=torch.float32),
            "label": torch.from_numpy(label),
            "mask": torch.from_numpy(mask),
            "os_time_days": torch.tensor(float(row["os_time_days"]), dtype=torch.float32),
            "os_event": torch.tensor(int(row["os_event"]), dtype=torch.long),
        }


class M1TileDataset(Dataset):
    """Tile-level dataset for UNI/UNI v2 on-the-fly feature extraction with augmentation."""

    def __init__(self, tile_index: pd.DataFrame, transform=None):
        self.tile_index = tile_index.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.tile_index)

    def __getitem__(self, idx):
        row = self.tile_index.iloc[idx]
        image = Image.open(row["tile_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        coords = row[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        return {
            "image": image,
            "coords": torch.from_numpy(coords),
            "case_id": row["case_id"],
            "tile_path": row["tile_path"],
        }


# split은 전체 생존 event 기준으로 stratify합니다. horizon별 unknown은 loss mask에서 처리합니다.
train_valid_df, test_df = train_test_split(
    slide_df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=slide_df["os_event"],
)
train_df, valid_df = train_test_split(
    train_valid_df,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=train_valid_df["os_event"],
)

train_case_ids = set(train_df["case_id"])
valid_case_ids = set(valid_df["case_id"])
test_case_ids = set(test_df["case_id"])
assert len(train_case_ids & valid_case_ids) == 0
assert len(train_case_ids & test_case_ids) == 0
assert len(valid_case_ids & test_case_ids) == 0

train_dataset = M1SlideDataset(train_df, tile_index_df)
valid_dataset = M1SlideDataset(valid_df, tile_index_df)
test_dataset = M1SlideDataset(test_df, tile_index_df)

train_tile_dataset = M1TileDataset(tile_index_df[tile_index_df["case_id"].isin(train_case_ids)], transform=get_train_patch_transform())
valid_tile_dataset = M1TileDataset(tile_index_df[tile_index_df["case_id"].isin(valid_case_ids)], transform=get_eval_patch_transform())
test_tile_dataset = M1TileDataset(tile_index_df[tile_index_df["case_id"].isin(test_case_ids)], transform=get_eval_patch_transform())

split_df = slide_df[["dataset", "case_id", "os_time_days", "os_event", "known_horizon_count"] + HORIZON_NAMES + [f"mask_{n}" for n in HORIZON_NAMES]].copy()
split_df["split"] = "unused"
split_df.loc[split_df["case_id"].isin(train_case_ids), "split"] = "train"
split_df.loc[split_df["case_id"].isin(valid_case_ids), "split"] = "valid"
split_df.loc[split_df["case_id"].isin(test_case_ids), "split"] = "test"
split_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_horizon_case_splits.csv", index=False)

print("slide splits:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("tile splits:", len(train_tile_dataset), len(valid_tile_dataset), len(test_tile_dataset))
display(pd.crosstab(split_df["split"], split_df["os_event"]))

split_horizon_summary = []
for split_name, part in split_df.groupby("split"):
    row = {"split": split_name, "n_cases": len(part)}
    for name in HORIZON_NAMES:
        known = part[f"mask_{name}"].eq(1)
        row[f"{name}_known"] = int(known.sum())
        row[f"{name}_dead"] = int(part.loc[known, name].sum())
    split_horizon_summary.append(row)
display(pd.DataFrame(split_horizon_summary))

sample = train_dataset[0]
print("sample case:", sample["case_id"])
print("n_tiles:", len(sample["tile_paths"]))
print("coords:", sample["coords"].shape, sample["coords"][:3])
print("slide_size:", sample["slide_size"])
print("label:", sample["label"])
print("mask:", sample["mask"])
print("model output 해석: logits -> sigmoid(logits) * 100 = horizon별 사망위험 percent")


Load TCGA_PAAD metadata: 100%|██████████| 160/160 [00:00<00:00, 199.08it/s]


all_slide_df: (160, 18)
slide_df: (152, 18)
excluded_df: (8, 19)
tile_index_df: (36064, 28)


,horizon,known_n,dead_n,alive_n,unknown_n,dead_rate_known
0,dead_by_6m,152,19,133,0,0.125000
1,dead_by_12m,139,41,98,13,0.294964
2,dead_by_18m,122,67,55,30,0.549180
3,dead_by_24m,112,84,28,40,0.750000


,case_id,n_tiles,os_time_days,os_event,known_horizon_count,dead_by_6m,dead_by_12m,dead_by_18m,dead_by_24m,mask_dead_by_6m,mask_dead_by_12m,mask_dead_by_18m,mask_dead_by_24m
0,TCGA-2J-AAB4,239,729.0,0,3,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
2,TCGA-2J-AAB1,289,66.0,1,4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,TCGA-2J-AAB6,326,293.0,1,4,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,TCGA-2J-AABA,101,607.0,1,4,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
5,TCGA-2J-AAB9,321,627.0,1,4,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0


slide splits: 96 25 31
tile splits: 22953 6490 6621


os_event,0,1
split,,
test,12,19
train,37,59
valid,10,15


,split,n_cases,dead_by_6m_known,dead_by_6m_dead,dead_by_12m_known,dead_by_12m_dead,dead_by_18m_known,dead_by_18m_dead,dead_by_24m_known,dead_by_24m_dead
0,test,31,31,2,30,10,26,12,24,19
1,train,96,96,12,87,26,76,42,70,50
2,valid,25,25,5,22,5,20,13,18,15


sample case: TCGA-FB-AAPS
n_tiles: 30
coords: torch.Size([30, 6]) tensor([[0.2395, 0.1403, 0.2994, 0.2104, 0.1197, 0.1403],
        [0.3592, 0.1403, 0.4191, 0.2104, 0.1197, 0.1403],
        [0.4790, 0.1403, 0.5388, 0.2104, 0.1197, 0.1403]])
slide_size: tensor([33864., 28910.])
label: tensor([0., 0., 0., 0.])
mask: tensor([1., 0., 0., 0.])
model output 해석: logits -> sigmoid(logits) * 100 = horizon별 사망위험 percent


In [ ]:
# M1 모델 정의 및 loss/optimizer 구성
# 구조: frozen UNI/UNI v2 feature extractor + coordinate spatial embedding + attention MIL

import torch
from torch import nn

from scripts.models.m1_pathology_mil import (
    M1ModelConfig,
    PathologySpatialMIL,
    masked_bce_with_logits,
    sample_tiles,
    build_optimizer,
)

if "device" not in globals():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

MAX_TILES_PER_SLIDE = 256
FEATURE_BATCH_SIZE = 32
M1_FEATURE_DIM = 1024  # UNI ViT-L 계열 기본 feature dim. UNI v2 사용 시 실제 출력 dim에 맞게 수정하세요.
SPATIAL_DIM = 128
FUSION_DIM = 512
MIL_HIDDEN_DIM = 256
DROPOUT = 0.25
N_OUTPUTS = len(HORIZON_NAMES)
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4


def compute_horizon_pos_weight(train_manifest: pd.DataFrame, horizon_names: list[str], device: torch.device) -> torch.Tensor:
    """BCEWithLogitsLoss용 horizon별 positive class weight 계산.

    unknown(mask=0)은 분모에서 제외합니다.
    pos_weight = n_negative / n_positive
    """
    weights = []
    rows = []
    for name in horizon_names:
        mask_col = f"mask_{name}"
        known = train_manifest[mask_col].eq(1)
        pos = float(train_manifest.loc[known, name].sum())
        neg = float(known.sum() - pos)
        weight = neg / max(pos, 1.0)
        weights.append(weight)
        rows.append({"horizon": name, "known": int(known.sum()), "positive_dead": int(pos), "negative_alive": int(neg), "pos_weight": weight})
    pos_weight_df = pd.DataFrame(rows)
    display(pos_weight_df)
    return torch.tensor(weights, dtype=torch.float32, device=device)


pos_weight = compute_horizon_pos_weight(train_df, HORIZON_NAMES, device)

m1_config = M1ModelConfig(
    feature_dim=M1_FEATURE_DIM,
    coord_dim=6,
    spatial_dim=SPATIAL_DIM,
    fusion_dim=FUSION_DIM,
    mil_hidden_dim=MIL_HIDDEN_DIM,
    n_outputs=N_OUTPUTS,
    dropout=DROPOUT,
    max_tiles=MAX_TILES_PER_SLIDE,
    feature_batch_size=FEATURE_BATCH_SIZE,
    freeze_feature_extractor=True,
)


def m1_loss_fn(logits: torch.Tensor, labels: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
    return masked_bce_with_logits(
        logits=logits,
        labels=labels,
        masks=masks,
        pos_weight=pos_weight,
    )


def initialize_m1_model(
    feature_extractor: nn.Module,
    feature_dim: int = M1_FEATURE_DIM,
    lr: float = LEARNING_RATE,
    weight_decay: float = WEIGHT_DECAY,
) -> tuple[PathologySpatialMIL, torch.optim.Optimizer]:
    config = M1ModelConfig(
        feature_dim=feature_dim,
        coord_dim=6,
        spatial_dim=SPATIAL_DIM,
        fusion_dim=FUSION_DIM,
        mil_hidden_dim=MIL_HIDDEN_DIM,
        n_outputs=N_OUTPUTS,
        dropout=DROPOUT,
        max_tiles=MAX_TILES_PER_SLIDE,
        feature_batch_size=FEATURE_BATCH_SIZE,
        freeze_feature_extractor=True,
    )
    model = PathologySpatialMIL(feature_extractor=feature_extractor, config=config).to(device)
    optimizer = build_optimizer(model, lr=lr, weight_decay=weight_decay)
    return model, optimizer


model = None
optimizer = None

if "feature_extractor" in globals():
    model, optimizer = initialize_m1_model(feature_extractor, feature_dim=M1_FEATURE_DIM)
    print("M1 model initialized with feature_extractor.")
    print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
else:
    print("feature_extractor가 아직 없습니다. 다음 셀에서 UNI/UNI v2를 로드한 뒤 아래처럼 실행하세요:")
    print("model, optimizer = initialize_m1_model(feature_extractor, feature_dim=M1_FEATURE_DIM)")

print("MAX_TILES_PER_SLIDE:", MAX_TILES_PER_SLIDE)
print("FEATURE_BATCH_SIZE:", FEATURE_BATCH_SIZE)
print("N_OUTPUTS:", N_OUTPUTS, HORIZON_NAMES)

# train loop에서 사용할 loss 예시:
# outputs = model(tile_images, coords)
# loss = m1_loss_fn(outputs["logits"], labels, masks)
# risk_percent = outputs["risk_percent"]
